# 03 · Closed-form chi-squared, cross-validated / 閉式卡方 + 交叉驗證

Goal: confirm that closed-form and Monte Carlo agree on white noise, then show the closed-form is **orders of magnitude faster**.

目標：驗證閉式解與 Monte Carlo 在白雜訊下結果一致，然後展示閉式解**快上好幾個量級**。

In [ ]:
import sys, pathlib, time
sys.path.append(str(pathlib.Path('..').resolve()))

import numpy as np
import matplotlib.pyplot as plt

from emdsig import chi2_confidence_bounds, monte_carlo_bounds, calibrate_intercept

## 1. Monte Carlo baseline (slow, but ground truth)

In [ ]:
N = 1024
t0 = time.perf_counter()
grid_mc, lo_mc, hi_mc = monte_carlo_bounds(
    N=N, n_trials=100, noise='white', seed=1, alpha=0.05,
)
mc_time = time.perf_counter() - t0
print(f'Monte Carlo (M=100) took {mc_time:.1f} s')

## 2. Closed-form chi-squared

In [ ]:
# Anchor c using the first Monte Carlo point (highest-frequency IMF).
order = np.argsort(grid_mc)
c_anchor = calibrate_intercept(grid_mc[order][0], 0.5 * (lo_mc + hi_mc)[order][0])

t0 = time.perf_counter()
grid_cf = np.linspace(grid_mc.min(), grid_mc.max(), 200)
lo_cf, hi_cf = chi2_confidence_bounds(grid_cf, N=N, alpha=0.05, baseline_intercept=c_anchor)
cf_time = time.perf_counter() - t0
print(f'Closed-form took {cf_time*1000:.2f} ms  — speedup {mc_time/cf_time:.0f}x')

## 3. Overlay

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
ax.fill_between(grid_cf, lo_cf, hi_cf, color='steelblue', alpha=0.25, label='Closed-form 95% CI')
ax.plot(grid_mc[order], lo_mc[order], 'o--', color='crimson', ms=4, label='MC lower')
ax.plot(grid_mc[order], hi_mc[order], 'o--', color='crimson', ms=4, label='MC upper')
ax.set_xlabel('ln(T)'); ax.set_ylabel('ln(E)')
ax.set_title('White noise: closed-form vs Monte Carlo, 95% CI')
ax.legend(); ax.grid(True, alpha=0.3)
plt.show()

## 4. Quantitative agreement / 定量比對

Expected: upper-bound disagreement below 5%. Larger gaps suggest too few MC trials or unstable EMD.

In [ ]:
hi_cf_at_mc = np.interp(grid_mc[order], grid_cf, hi_cf)
gap = np.abs(hi_mc[order] - hi_cf_at_mc)
print(f'max |MC_upper - CF_upper|: {gap.max():.3f}  (natural log scale)')
print(f'relative (exp of gap): {(np.exp(gap) - 1).max() * 100:.1f}%')